# Imports & Environnement

In [2]:
import os
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader 
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from IPython.display import Markdown, display

# Initialisation du LLM

In [3]:
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

# Chemin vers mon document PDF

In [21]:
# Je laisse cette partie vide avant de la mettre dans le classroom pour plus de sécurité
pdf_file = "./src/ESSOAZINA_ANGE_RESUME.pdf"

# Chargement du PDF

In [24]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

# 1. Vérifie si pdfs/ est au niveau courant ou au niveau parent
possible_paths = [
    Path("pdfs/ESSOAZINA_ANGE_RESUME.pdf"),
    Path("../pdfs/ESSOAZINA_ANGE_RESUME.pdf")
]

pdf_path = next((p for p in possible_paths if p.is_file()), None)

if pdf_path is None:
    raise FileNotFoundError(
        f"Fichier introuvable. Assurez-vous que le PDF est bien présent dans le dossier 'pdfs/'."
    )

print(f"Chargement depuis : {pdf_path.resolve()}")

# 2. Charge le document
pdf_loader = PyPDFLoader(str(pdf_path))
docs = pdf_loader.load()
print(f"Nombre de pages chargées : {len(docs)}")

Chargement depuis : C:\Users\essoa\Downloads\RAG\pdfs\ESSOAZINA_ANGE_RESUME.pdf
Nombre de pages chargées : 2


 Découpage (Text Splitter)

In [25]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""]
)

 Application du Splitter et comptage des chunks

In [26]:
chunks = pdf_loader.load_and_split(text_splitter)
print("Nombre de chunks générés :", len(chunks))

Nombre de chunks générés : 5


# Modèle d'Embedding Open Source

In [27]:
embedding_model = OllamaEmbeddings(
    model="all-minilm"
)

 Vector Store (ChromaDB)

In [29]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="mon_cv_rag",
    persist_directory="./store"
)

# Définition du Retriever

In [30]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

Test de recherche de Chunks

In [34]:
user_input = "Quelles sont les compétences et technologies maîtrisées par le candidat ?"

retrieved_chunks = retriever.invoke(user_input)
print(retrieved_chunks)
print("Nombre de chunks récupérés :", len(retrieved_chunks))

[Document(metadata={'keywords': '', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'total_pages': 2, 'page': 1, 'page_label': '2', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-12T11:10:53+00:00', 'producer': 'pdfTeX-1.40.27', 'title': '', 'subject': '', 'source': '..\\pdfs\\ESSOAZINA_ANGE_RESUME.pdf', 'moddate': '2026-08-12T11:10:53+00:00', 'author': ''}, page_content='TECHNICAL SKILLS\nML / AIScikit-learn·PyTorch·TensorFlow Lite·HuggingFace Transformers·LLM Fine-tuning\nML MethodsIsolation Forest·SVM·Random Forest·CNN·ANN·Autoencoders·Federated Learning\nBig DataApache Kafka·Hadoop·MongoDB·Neo4j·OrientDB·MariaDB\nLanguagesPython (primary)·Java·JavaScript·SQL\nFull-StackFastAPI·Spring Boot·Next.js·Tailwind CSS\nToolsGit·Docker (basic)·Linux·AWS (in progress)·Jupyter\nIoTRaspberry Pi·Sensor integration·Edge inference (TFLite)\nACHIEVEMENTS & LANGUAGES\nAchievements Languages\nPFE Fraud Detec

# Prompt Template

In [35]:
prompt_template = """
Answer the following question based only on provided context.
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!

<context>
{context}
</context>
<question>
{question}
</question>
"""

 Préparation et concaténation du contexte

In [36]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

print("Nombre de chunks pertinents :", len(relevant_document_chunks))

Nombre de chunks pertinents : 3


 Formatage du prompt

In [37]:
prompt = prompt_template.format(context=context_for_query, question=user_input)
print(prompt)


Answer the following question based only on provided context.
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!

<context>
TECHNICAL SKILLS
ML / AIScikit-learn·PyTorch·TensorFlow Lite·HuggingFace Transformers·LLM Fine-tuning
ML MethodsIsolation Forest·SVM·Random Forest·CNN·ANN·Autoencoders·Federated Learning
Big DataApache Kafka·Hadoop·MongoDB·Neo4j·OrientDB·MariaDB
LanguagesPython (primary)·Java·JavaScript·SQL
Full-StackFastAPI·Spring Boot·Next.js·Tailwind CSS
ToolsGit·Docker (basic)·Linux·AWS (in progress)·Jupyter
IoTRaspberry Pi·Sensor integration·Edge inference (TFLite)
ACHIEVEMENTS & LANGUAGES
Achievements Languages
PFE Fraud Detection — 18/20, top of cohort
Top of region, national baccalauréat
Nazahathon hackathon — blockchain transparency
project
French — Native
English — C2 Proficient (professional working level)
Italian — A2 Beginner. ESSOAZINA ANGE RAIY ANE
AI / Machin

# Génération de la réponse par le LLM

In [42]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:1b",  # ou le modèle que tu utilises
    num_ctx=1024,     # Réduit le buffer mémoire (par défaut souvent 2048+)
    temperature=0
)

# Fonction RAG globale

In [43]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query = ". ".join(context_list)   
    prompt = prompt_template.format(context=context_for_query, question=query)
    resp = llm.invoke(prompt)
    return resp.content

Test anti-hallucination

In [45]:
resp_hors_sujet = RAG("Quels sont les événements culturels dans le sud-est du Maroc ?")
display(Markdown(resp_hors_sujet))

Je ne sais pas!

# Préparation de l'évaluation

In [46]:
eval_question = "Quelles sont les compétences et technologies maîtrisées par le candidat ?"

relevant_document_chunks = retriever.invoke(eval_question)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

user_message_template = """
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

answer = RAG(eval_question)
display(Markdown(answer))

Selon le context, les compétences et technologies maîtrisées par le candidat sont :

* Machine learning (ML) et deep learning
* Développement de systèmes d'apprentissage automatique (Machine Learning Engineer)
* Développement de systèmes d'apprentissage automatique (Data Analyst)
* Développement de systèmes d'apprentissage automatique (AI Community Agent)
* Développement de systèmes d'apprentissage automatique (Computer Vision et NLP)
* Développement de systèmes d'apprentissage automatique (IoT)
* Développement de systèmes d'apprentissage automatique (FastAPI)
* Développement de systèmes d'apprentissage automatique (PyTorch)
* Développement de systèmes d'apprentissage automatique (TensorFlow Lite)
* Développement de systèmes d'apprentissage automatique (HuggingFace Transformers)
* Développement de systèmes d'apprentissage automatique (LLM)
* Développement de systèmes d'apprentissage automatique (Big Data)
* Développement de systèmes d'apprentissage automatique (Apache Kafka)
* Développement de systèmes d'apprentissage automatique (Hadoop)
* Développement de systèmes d'apprentissage automatique (MongoDB)
* Développement de systèmes d'apprentissage automatique (Neo4j)
* Développement de systèmes d'apprentissage automatique (OrientDB)
* Développement de systèmes d'apprentissage automatique (OpenCV)
* Développement de systèmes d'apprentissage automatique (Raspberry Pi)
* Développement de systèmes d'apprentissage automatique (FastAPI)
* Développement de systèmes d'apprentissage automatique (HuggingFace Transformers)
* Développement de systèmes d'apprentissage automatique (PyTorch)
* Développement de systèmes d'apprentissage automatique (TensorFlow Lite)
* Développement de systèmes d'apprentissage automatique (LLM)
* Développement de systèmes d'apprentissage automatique (Machine Learning)
* Développement de systèmes d'apprentissage automatique (Computer Vision)
* Développement de systèmes d'apprentissage automatique (NLP)
* Développement de systèmes d'apprentissage automatique (Medical Chatbot)
* Développement de systèmes d'apprentissage automatique (IoT)

LLM Juge (Groundness / Fidélité)

In [47]:
groundness_rater_system_message = """
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

groundness_checker = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

Fonction d'évaluation & Exécution du Juge

In [48]:
def evaluate(system_message, user_message_template, question, model=groundness_checker):
    retrieved_chunks = retriever.invoke(question)
    context_list = [d.page_content for d in retrieved_chunks]
    context = ". ".join(context_list)
    answer = RAG(question) 
    prompt = f"""
{system_message}

USER:
{user_message_template.format(question=question, context=context, answer=answer)}
"""
    juge_response = model.invoke(prompt)
    return juge_response.content

# Lancement de l'évaluation
verdict = evaluate(groundness_rater_system_message, user_message_template, eval_question)
display(Markdown(verdict))

Étapes nécessaires pour évaluer la réponse selon la métrique :

1. Lire attentivement la question posée par l'utilisateur et le contexte fourni.
2. Identifier les informations pertinentes contenues dans le contexte qui sont liées à la question posée.
3. Analyser la réponse générée par l'IA pour voir si elle est basée uniquement sur les informations présentées dans le contexte.
4. Vérifier si la réponse contient des informations qui ne sont pas directement liées à la question posée ou qui sont extraites de sources externes.
5. Évaluer la réponse en fonction de la métrique requise, en considérant si la réponse est dérivée uniquement des informations présentées dans le contexte.

Évaluation de la métrique :

La réponse générée par l'IA contient des informations qui sont directement liées à la question posée, notamment les compétences et technologies maîtrisées par le candidat. Cependant, la réponse contient également des informations qui ne sont pas directement liées à la question, telles que les informations sur les projets personnels et les expériences professionnelles du candidat.

La réponse contient des informations qui sont extraites de sources externes, telles que les informations sur les projets personnels et les expériences professionnelles du candidat. Cela signifie que la réponse n'est pas basée uniquement sur les informations présentées dans le contexte.

Évaluation selon les critères d'évaluation :

La métrique n'est pas respectée dans une grande partie, car la réponse contient des informations qui ne sont pas directement liées à la question posée et qui sont extraites de sources externes. Cependant, la réponse contient également des informations qui sont directement liées à la question, ce qui signifie que la métrique est respectée dans une certaine mesure.

Score : 2/5

La réponse n'est pas entièrement respectée de la métrique, car elle contient des informations qui ne sont pas directement liées à la question posée et qui sont extraites de sources externes. Cependant, la réponse contient également des informations qui sont directement liées à la question, ce qui signifie que la métrique est respectée dans une certaine mesure.